## Setup

In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt 
import seaborn as sns
import numpy as np
import matplotlib.cm as cm
import scipy.stats
import statsmodels.api as sm

# Read all CSV files from the specified directory
csv_files = glob.glob("data/txs/august/*_pnl.csv")
txs = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
auctions = pd.read_csv("data/auctions/timeboost_auctions.csv")

if 'merged' in txs.columns:
    txs = txs.drop(columns=['merged', 'auction_round', 'winner_address','winner_name','top_bid_eth','paid_bid_eth', 'express_lane_controller_address'])
    
# 1. Ensure proper datetime types and remove tz
txs["block_time"] = pd.to_datetime(txs["block_time"], utc=True).dt.tz_convert(None)
auctions["round_start_time"] = pd.to_datetime(auctions["round_start_time"], utc=True)
auctions["round_end_time"]   = pd.to_datetime(auctions["round_end_time"], utc=True)

# Take only auctions in August 2025
auctions = auctions[(auctions["round_start_time"] >= "2025-08-01") & (auctions["round_end_time"] < "2025-09-01")]

# 2. Sort both frames
txs = txs.sort_values("block_time")
auctions = auctions.sort_values("round_start_time")


In [ ]:
binance_btcusd = pd.read_csv("data/prices/BTCUSDT-1s-2025-08.csv")
binance_ethusd = pd.read_csv("data/prices/ETHUSDT-1s-2025-08.csv")
binance_arbusd = pd.read_csv("data/prices/ARBUSDT-1s-2025-08.csv")

cols = [
    "open_time_us", "open", "high", "low", "close", "volume",
    "close_time_us", "quote_vol", "trades", "taker_base", "taker_quote", "ignore"
]

binance_arbusd.columns = cols
binance_ethusd.columns = cols
binance_btcusd.columns = cols

horizons = [0, 5, 10, 300, 600, 1800]

def prepare_pricefeed(df, token):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["open_time_us"] / 1e6, unit="s", utc=True)
    df["midprice"] = (df["high"] + df["low"]) / 2
    df = df[["timestamp", "midprice"]].sort_values("timestamp").reset_index(drop=True)
    df = df.rename(columns={"midprice": f"{token}_mid"})
    return df

binance_arbusd  = prepare_pricefeed(binance_arbusd, "ARB")
binance_ethusd  = prepare_pricefeed(binance_ethusd, "ETH")
binance_btcusd  = prepare_pricefeed(binance_btcusd, "BTC")

In [ ]:
eth = binance_ethusd.sort_values("timestamp").copy()
eth["log_ret"] = np.log(eth["ETH_mid"] / eth["ETH_mid"].shift(1))

# 30-min rolling window (1800 seconds)
eth["vol_30m"] = eth["log_ret"].rolling(1800).std()

# Daily volatility = mean of intraday rolling vol
daily_vol = (
    eth.assign(date=eth["timestamp"].dt.date)
       .groupby("date")["vol_30m"]
       .mean()
       .reset_index()
       .rename(columns={"vol_30m": "daily_vol"})
)


In [ ]:
# Calculate USD for top_bid_eth and paid_bid_eth, using the eth price at auction start time
auctions = auctions.merge(
    binance_ethusd.rename(columns={"ETH_mid": "eth_price_at_auction_start"}),
    left_on="round_start_time",
    right_on="timestamp",
    how="left"
)
auctions["top_bid_usd"]  = auctions["top_bid_eth"]  * auctions["eth_price_at_auction_start"]
auctions["paid_bid_usd"] = auctions["paid_bid_eth"] * auctions["eth_price_at_auction_start"]

# Update txs with auction info
txs = txs.merge(
    auctions[["auction_round", "top_bid_usd", "paid_bid_usd"]],
    on="auction_round",
    how="left"
)

In [ ]:
#  Plot volatility over time
plt.figure(figsize=(12, 6))
plt.plot(daily_vol["date"], daily_vol["daily_vol"], label="Volatility", color='blue')
plt.xlabel("Time")
plt.ylabel("Volatility")
plt.title("Daily 30-Minute Rolling Volatility of ETH/USD")
plt.legend()
plt.grid()
plt.show()


In [ ]:
SEARCHERS = {
    "Wintermute": [
        '0xcb43d843f6cadf4f4844f3f57032468aadd9b95c',
        '0x27920e8039d2b6e93e36f5d5f53b998e2e631a70'
    ],
    "Selini": [
        '0xee2e7bbb67676292af2e31dffd1fea2276d6c7ba'
    ]
}

wm_txs = txs[txs["tx_to_address"].isin(SEARCHERS["Wintermute"])]
sel_txs = txs[txs["tx_to_address"].isin(SEARCHERS["Selini"])]

wm_txs["date"] = wm_txs["block_time"].dt.date
sel_txs["date"] = sel_txs["block_time"].dt.date

PnL Calculation (if needed)

In [ ]:
# txs = pd.merge_asof(
#     txs,
#     auctions[["auction_round", "round_start_time", "round_end_time"]],
#     left_on="block_time",
#     right_on="round_start_time",
#     direction="backward"
# )

# txs = txs[txs["block_time"] < txs["round_end_time"]].copy()

# txs["time_since_round_start"] = (
#     txs["block_time"] - txs["round_start_time"]
# ).dt.total_seconds()

# # Merge with auctions on auction_round to get more auction details
# txs = txs.merge(
#     auctions.drop(columns=["round_start_time", "round_end_time"]),
#     on="auction_round",
#     how="left"
# )

# txs["block_time"] = pd.to_datetime(txs["block_time"], utc=True)
# txs = txs.sort_values("block_time").reset_index(drop=True)

# def merge_markouts(sample, pricefeed, token, horizons=horizons):
#     out = sample.copy()

#     # PRICEFEED must not have duplicate timestamps or unnamed junk
#     pf = pricefeed[["timestamp", f"{token}_mid"]].copy().sort_values("timestamp")

#     for h in horizons:
#         mark_col = f"{token}_mark_t{h}"
#         price_col = f"{token}_mid_t{h}"

#         # target timestamp for markout
#         out[mark_col] = out["block_time"] + pd.to_timedelta(h, unit="s")

#         # IMPORTANT: drop leftover timestamps from previous merges
#         out = out.drop(columns=["timestamp"], errors="ignore")

#         merged = pd.merge_asof(
#             out.sort_values(mark_col),
#             pf,
#             left_on=mark_col,
#             right_on="timestamp",
#             direction="backward"
#         )

#         # Rename the merged midprice
#         merged = merged.rename(columns={f"{token}_mid": price_col})

#         # Drop redundant timestamp column from pricefeed
#         merged = merged.drop(columns=["timestamp"], errors="ignore")

#         out = merged  # continue loop

#     return out

# sample_txs = txs.copy()

# sample_txs = merge_markouts(sample_txs, binance_arbusd, "ARB")
# sample_txs = merge_markouts(sample_txs, binance_ethusd, "ETH")
# sample_txs = merge_markouts(sample_txs, binance_btcusd, "BTC")

# stablecoins = {"USDC", "USD₮0"}

# def get_price(token_symbol, row, h):
#     """
#     Returns the USD price for a token at horizon h
#     """
#     tok = token_symbol.replace("W", "")  # WETH→ETH, WBTC→BTC

#     # Stablecoins: fixed $1
#     if tok in stablecoins:
#         return 1.0

#     col = f"{tok}_mid_t{h}"
#     return row[col] if col in row and pd.notna(row[col]) else None


# def compute_pnl_row(row, h):
#     buy_tok  = row["bought_token_symbol"]
#     sell_tok = row["sold_token_symbol"]

#     buy_price  = get_price(buy_tok, row, h)
#     sell_price = get_price(sell_tok, row, h)

#     # Fees in USD
#     eth_fee_usd = row["tx_fee_eth"] * row["ETH_mid_t0"]

#     # If feed missing → cannot compute PNL
#     if buy_price is None or sell_price is None:
#         return None

#     pnl = (
#         row["bought_token_amount"] * buy_price - 
#         row["sold_token_amount"] * sell_price - 
#         eth_fee_usd
#     )
#     return pnl


# # Compute PNL for each horizon
# for h in horizons:
#     sample_txs[f"pnl_t{h}"] = sample_txs.apply(lambda row: compute_pnl_row(row, h), axis=1)
    
# logfile = "daily_pnl_log.txt"

# with open(logfile, "w") as f:

#     def log(msg):
#         print(msg)
#         f.write(msg + "\n")

#     daily_pnl = {}
#     sample_txs["date"] = sample_txs["block_time"].dt.date

#     for h in horizons:
#         col = f"pnl_t{h}"

#         # keep only txs with positive pnl at this horizon
#         df_pos = sample_txs[sample_txs[col] > 0]

#         df = (
#             df_pos.groupby(["date", "timeboosted"])[col]
#             .sum()
#             .reset_index()
#             .rename(columns={col: "pnl"})
#         )

#         daily_pnl[h] = df

#         # find avg pnl and count per timeboosted status across all txs
#         total_timeboosted = df_pos[df_pos["timeboosted"] == True][col].sum()
#         total_not_timeboosted = df_pos[df_pos["timeboosted"] == False][col].sum()
#         avg_timeboosted = df_pos[df_pos["timeboosted"] == True][col].mean()
#         avg_not_timeboosted = df_pos[df_pos["timeboosted"] == False][col].mean()
#         timeboosted_count = df_pos[df_pos["timeboosted"] == True].shape[0]
#         not_timeboosted_count = df_pos[df_pos["timeboosted"] == False].shape[0]

#         log("Positive PNL Daily Summary:")
#         log(f"Markout {h} seconds:")
#         log(f"  Total PNL Timeboosted: ${total_timeboosted:,.2f}")
#         log(f"  Total PNL Not Timeboosted: ${total_not_timeboosted:,.2f}")
#         log(f"  Average PNL Timeboosted: ${avg_timeboosted:,.2f} over {timeboosted_count} trades")
#         log(f"  Average PNL Not Timeboosted: ${avg_not_timeboosted:,.2f} over {not_timeboosted_count} trades")
#         log("\n" + "-"*60 + "\n")
        
#         # repeat for all PNL (not just positive)
#         total_timeboosted = sample_txs[sample_txs["timeboosted"] == True][col].sum()
#         total_not_timeboosted = sample_txs[sample_txs["timeboosted"] == False][col].sum()
#         avg_timeboosted = sample_txs[sample_txs["timeboosted"] == True][col].mean()
#         avg_not_timeboosted = sample_txs[sample_txs["timeboosted"] == False][col].mean()
#         timeboosted_count = sample_txs[sample_txs["timeboosted"] == True].shape[0]
#         not_timeboosted_count = sample_txs[sample_txs["timeboosted"] == False].shape[0]
        
#         log("All PNL Daily Summary:")
#         log(f"Markout {h} seconds:")
#         log(f"  Total PNL Timeboosted: ${total_timeboosted:,.2f}")
#         log(f"  Total PNL Not Timeboosted: ${total_not_timeboosted:,.2f}")
#         log(f"  Average PNL Timeboosted: ${avg_timeboosted:,.2f} over {timeboosted_count} trades")
#         log(f"  Average PNL Not Timeboosted: ${avg_not_timeboosted:,.2f} over {not_timeboosted_count} trades")
#         log("\n" + "-"*60 + "\n")

# print(f"Logs written to {logfile}")



## Daily

In [ ]:
auctions["date"] = auctions["round_start_time"].dt.date
auctions["bid_gap"] = auctions["top_bid_usd"] - auctions["paid_bid_usd"]
daily_bid_gap = auctions.groupby("date")["bid_gap"].mean().reset_index()
daily_gap_vol = daily_bid_gap.merge(daily_vol, on="date", how="inner")
df = daily_gap_vol.dropna(subset=["bid_gap", "daily_vol"])

pearson_r, pearson_p = scipy.stats.pearsonr(df["bid_gap"], df["daily_vol"])
spearman_r, spearman_p = scipy.stats.spearmanr(df["bid_gap"], df["daily_vol"])

print("Correlation between Daily Bid Gap and Daily Volatility:")
print(f"Pearson:  r={pearson_r:.4f}, p={pearson_p:.4g}")
print(f"Spearman: r={spearman_r:.4f}, p={spearman_p:.4g}")

In [ ]:
def compute_daily_pnl_gaps(txs, daily_vol):
    """
    txs: filtered tx dataset for a single actor (wm_txs, sel_txs, ...)
    daily_vol: dataframe with "date" and "daily_vol"
    
    Returns: dict of {horizon → merged df with tb, ntb, gap, vol}
    """

    # Ensure date exists
    if "date" not in txs:
        txs["date"] = txs["block_time"].dt.date

    daily_summary = {}

    for h in horizons:
        col = f"pnl_t{h}"

        # Only positive PnL rows
        df_pos = txs[txs[col] > 0]

        # Group by day + TB status
        df = (
            df_pos.groupby(["date", "timeboosted"])[col]
            .sum()
            .reset_index()
            .rename(columns={col: f"pnl_t{h}"})
        )

        # Pivot → two columns: TB and NTB
        pivot = (
            df.pivot(index="date", columns="timeboosted", values=f"pnl_t{h}")
              .rename(columns={
                  True:  f"tb_pnl_t{h}",
                  False: f"ntb_pnl_t{h}"
              })
              .reset_index()
        )

        # Add volatility
        merged = pivot.merge(daily_vol, on="date", how="left")

        # PnL advantage
        merged[f"gap_t{h}"] = merged[f"tb_pnl_t{h}"] - merged[f"ntb_pnl_t{h}"]

        daily_summary[h] = merged
    
    return daily_summary


def analyze_correlations(daily_summary):
    """Print correlations (Pearson + Spearman) per horizon."""
    for h, df in daily_summary.items():
        df2 = df.dropna(subset=["daily_vol", f"gap_t{h}"])

        pearson_r, pearson_p = scipy.stats.pearsonr(
            df2["daily_vol"], df2[f"gap_t{h}"]
        )
        spearman_r, spearman_p = scipy.stats.spearmanr(
            df2["daily_vol"], df2[f"gap_t{h}"]
        )

        print(f"Horizon {h}s:")
        print(f"  Pearson:  r={pearson_r:.4f}, p={pearson_p:.4g}")
        print(f"  Spearman: r={spearman_r:.4f}, p={spearman_p:.4g}")
        print("-" * 60)


def plot_scatter(daily_summary, title_prefix):
    """Scatter plots for each horizon."""
    for h, df in daily_summary.items():
        plt.figure(figsize=(8,5))
        plt.scatter(df["daily_vol"], df[f"gap_t{h}"], alpha=0.7)
        plt.title(f"{title_prefix}: Vol vs PnL Gap (h={h}s)")
        plt.xlabel("Daily Volatility")
        plt.ylabel(f"TB Advantage (PnL gap t={h}s)")
        plt.grid()
        plt.show()


In [ ]:
wm_summary = compute_daily_pnl_gaps(wm_txs, daily_vol)

# analyze_correlations(wm_summary)
# plot_scatter(wm_summary, "Wintermute")

sel_summary = compute_daily_pnl_gaps(sel_txs, daily_vol)

# analyze_correlations(sel_summary)
# plot_scatter(sel_summary, "Selini")


In [ ]:
def correlations_to_latex(wm_summary, sel_summary, horizons):
    """
    Returns a LaTeX tabular environment string comparing WM vs Selini.
    Columns: horizon, WM_r, WM_p, SEL_r, SEL_p
    """

    rows = []

    # Header
    header = (
        "\\begin{table}[h]\n"
        "\\centering\n"
        "\\begin{tabular}{c|cc|cc}\n"
        "\\toprule\n"
        "Horizon & WM r & WM p & Selini r & Selini p \\\\\n"
        "\\midrule\n"
    )

    for h in horizons:
        df_wm  = wm_summary[h].dropna(subset=["daily_vol", f"gap_t{h}"])
        df_sel = sel_summary[h].dropna(subset=["daily_vol", f"gap_t{h}"])

        # If not enough data
        if len(df_wm) < 2:
            wm_r, wm_p = float("nan"), float("nan")
        else:
            wm_r, wm_p = scipy.stats.pearsonr(df_wm["daily_vol"], df_wm[f"gap_t{h}"])

        if len(df_sel) < 2:
            sel_r, sel_p = float("nan"), float("nan")
        else:
            sel_r, sel_p = scipy.stats.pearsonr(df_sel["daily_vol"], df_sel[f"gap_t{h}"])

        # Format row
        row = (
            f"{h}s & "
            f"{wm_r:.3f} & {wm_p:.3f} & "
            f"{sel_r:.3f} & {sel_p:.3f} \\\\"
        )
        rows.append(row)

    footer = (
        "\\bottomrule\n"
        "\\end{tabular}\n"
        "\\caption{Correlation between daily volatility and TB-NonTB PnL gap for Wintermute and Selini.}\n"
        "\\label{tab:vol_pnl_corr}\n"
        "\\end{table}"
    )

    # Combine everything
    latex_table = header + "\n".join(rows) + "\n" + footer
    return latex_table


# Usage:
latex_code = correlations_to_latex(wm_summary, sel_summary, horizons)

with open("vol_corr_table.tex", "w") as f:
    f.write(latex_code)


In [ ]:
def plot_daily_pnl_grid(daily_summary, actor_name):
    """
    daily_summary: dict {h → df} created by compute_daily_pnl_gaps()
    actor_name: "Wintermute" or "Selini"
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    axes = axes.flatten()

    for i, h in enumerate(horizons):
        ax = axes[i]
        df = daily_summary[h].copy()

        # Columns created earlier:
        tb_col  = f"tb_pnl_t{h}"
        ntb_col = f"ntb_pnl_t{h}"

        ax.plot(df["date"], df[tb_col],  marker="o", color="blue",   label="Timeboosted")
        ax.plot(df["date"], df[ntb_col], marker="o", color="orange", label="Non-Timeboosted")

        ax.set_title(f"{h}s Horizon")
        ax.set_xlabel("Date")
        ax.set_ylabel("PNL (USD)")
        ax.grid(True)

        # Avoid overcrowded labels
        ax.xaxis.set_major_locator(plt.MaxNLocator(4))
        ax.legend()

    # Hide unused axes (none here, since horizons = 6)
    for j in range(len(horizons), len(axes)):
        axes[j].axis("off")

    fig.suptitle(f"{actor_name}: Daily Timeboosted vs Non-Timeboosted PnL", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

plot_daily_pnl_grid(wm_summary, "Wintermute")
plot_daily_pnl_grid(sel_summary, "Selini")

## PnL

In [ ]:
def compute_actor_summary(df, horizons):
    out = {}

    # Total counts
    out["tb_count"]  = df[df["timeboosted"] == True].shape[0]
    out["ntb_count"] = df[df["timeboosted"] == False].shape[0]

    # Total Volume
    out["tb_vol"]  = df[df["timeboosted"] == True]["amount_usd"].sum()
    out["ntb_vol"] = df[df["timeboosted"] == False]["amount_usd"].sum()
    
    # Compute total paid_in_usd
    out["tb_paid_usd"]  = auctions[auctions["auction_round"].isin(
        df[df["timeboosted"] == True]["auction_round"].unique()
    )]["paid_bid_usd"].sum()

    # Total PnL aggregated across all horizons
    for h in horizons:
        col = f"pnl_t{h}"
        
        df_pos = df[df[col] > 0]
        
        out[f"tb_pnl_t{h}"]  = df_pos[df_pos["timeboosted"] == True][col].sum()
        out[f"ntb_pnl_t{h}"] = df_pos[df_pos["timeboosted"] == False][col].sum()
        out[f"tb_avg_pnl_t{h}"]  = df_pos[df_pos["timeboosted"] == True][col].mean()
        out[f"ntb_avg_pnl_t{h}"] = df_pos[df_pos["timeboosted"] == False][col].mean()

    return out


wm_summary_stats  = compute_actor_summary(wm_txs, horizons)
sel_summary_stats = compute_actor_summary(sel_txs, horizons)


In [ ]:
def build_combined_table(wm_stats, sel_stats, horizons):
    rows = []

    # --- High-level totals ---
    rows.append(r"\textbf{Total TB Trades} & "
                f"{wm_stats['tb_count']:,} & {sel_stats['tb_count']:,} \\\\")
    rows.append(r"\textbf{Total Non-TB Trades} & "
                f"{wm_stats['ntb_count']:,} & {sel_stats['ntb_count']:,} \\\\")
    rows.append(r"\textbf{Total TB Volume (USD)} & "
                f"${wm_stats['tb_vol']:,.2f}$ & ${sel_stats['tb_vol']:,.2f}$ \\\\")
    rows.append(r"\textbf{Total Non-TB Volume (USD)} & "
                f"${wm_stats['ntb_vol']:,.2f}$ & ${sel_stats['ntb_vol']:,.2f}$ \\\\")
    rows.append(r"\textbf{Total Auction Bids Paid (USD)} & "
                f"${wm_stats['tb_paid_usd']:,.2f}$ & ${sel_stats['tb_paid_usd']:,.2f}$ \\\\")

    rows.append(r"\midrule")

    # --- Horizon-level PnL & Avg PnL ---
    for h in horizons:
        # Total PnL row
        rows.append(
            rf"\textbf{{TB PnL @ {h}s}} & "
            f"${wm_stats[f'tb_pnl_t{h}']:,.2f}$ & "
            f"${sel_stats[f'tb_pnl_t{h}']:,.2f}$ \\\\"
        )
        # Average below it (not bloating: subtle second line)
        rows.append(
            rf"\quad Avg PnL & "
            f"${wm_stats[f'tb_avg_pnl_t{h}']:,.2f}$ & "
            f"${sel_stats[f'tb_avg_pnl_t{h}']:,.2f}$ \\\\"
        )

        rows.append(
            rf"\textbf{{Non-TB PnL @ {h}s}} & "
            f"${wm_stats[f'ntb_pnl_t{h}']:,.2f}$ & "
            f"${sel_stats[f'ntb_pnl_t{h}']:,.2f}$ \\\\"
        )
        rows.append(
            rf"\quad Avg PnL & "
            f"${wm_stats[f'ntb_avg_pnl_t{h}']:,.2f}$ & "
            f"${sel_stats[f'ntb_avg_pnl_t{h}']:,.2f}$ \\\\"
        )

        rows.append(r"\addlinespace")

    latex = r"""
\begin{table}[ht]
\centering
\begin{tabular}{l rr}
\toprule
\textbf{Metric} & \textbf{Wintermute} & \textbf{Selini} \\
\midrule
""" + "\n".join(rows) + r"""
\bottomrule
\end{tabular}
\caption{Comparison of Wintermute and Selini Timeboosted vs Non-Timeboosted PnL, Averages, Volumes, and Trade Counts.}
\end{table}
"""

    return latex


combined_latex = build_combined_table(wm_summary_stats, sel_summary_stats, horizons)
with open("actor_comparison.tex", "w") as f:
    f.write(combined_latex)

In [ ]:
def compute_entity_round_pnl(df, horizons, winner):
    """
    Compute PNL per auction round for a single entity (Wintermute or Selini),
    broken into timeboosted and non-timeboosted, for all horizons.
    """

    results = []
    
    df = df[df["winner_name"] == winner]

    # group by auction round
    for round_id, g in df.groupby("auction_round"):

        row = {
            "auction_round": round_id,
            "winner_name": g["winner_name"].iloc[0],
            "top_bid_usd": g["top_bid_usd"].iloc[0],
            "paid_bid_usd": g["paid_bid_usd"].iloc[0],
            "round_start_time": g["round_start_time"].iloc[0],
            "round_end_time": g["round_end_time"].iloc[0],
        }

        for h in horizons:
            col = f"pnl_t{h}"

            # g_pos = g[g[col] > 0]   # optional: only positive pnl trades

            tb  = g[g["timeboosted"] == True]

            row[f"tb_pnl_t{h}"]  = tb[col].sum()

        results.append(row)

    return pd.DataFrame(results)


In [ ]:
wm_round_pnl = compute_entity_round_pnl(wm_txs, horizons, "Wintermute")
sel_round_pnl = compute_entity_round_pnl(sel_txs, horizons, "Selini")


In [ ]:
def compute_corr(df, bid_col, pnl_col="tb_pnl_t5"):
    df2 = df.dropna(subset=[bid_col, pnl_col])
    pear_r, pear_p = scipy.stats.pearsonr(df2[bid_col], df2[pnl_col])
    spear_r, spear_p = scipy.stats.spearmanr(df2[bid_col], df2[pnl_col])
    return pear_r, pear_p, spear_r, spear_p


corr_results = {
    "Wintermute": {
        "paid": compute_corr(wm_round_pnl, "paid_bid_usd"),
        "top":  compute_corr(wm_round_pnl, "top_bid_usd"),
    },
    "Selini": {
        "paid": compute_corr(sel_round_pnl, "paid_bid_usd"),
        "top":  compute_corr(sel_round_pnl, "top_bid_usd"),
    }
}

latex_table = r"""
\begin{table}[h!]
\centering
\renewcommand{\arraystretch}{1.3}
\begin{tabular}{lcccccccc}
\toprule
& \multicolumn{4}{c}{\textbf{Paid Bid vs PnL}} 
& \multicolumn{4}{c}{\textbf{Top Bid vs PnL}} \\
\cmidrule(lr){2-5} \cmidrule(lr){6-9}
\textbf{Entity} 
& \textbf{P-r} & \textbf{P-p} & \textbf{S-r} & \textbf{S-p}
& \textbf{P-r} & \textbf{P-p} & \textbf{S-r} & \textbf{S-p} \\
\midrule
Wintermute 
& %.4f & %.2g & %.4f & %.2g
& %.4f & %.2g & %.4f & %.2g \\
Selini 
& %.4f & %.2g & %.4f & %.2g
& %.4f & %.2g & %.4f & %.2g \\
\bottomrule
\end{tabular}
\caption{Correlation between auction bids (paid and top) and realized PnL for 5-second horizon.}
\end{table}
""" % (
    *corr_results["Wintermute"]["paid"],
    *corr_results["Wintermute"]["top"],
    *corr_results["Selini"]["paid"],
    *corr_results["Selini"]["top"]
)
with open("bid_pnl_corr.tex", "w") as f:
    f.write(latex_table)


In [ ]:
def corr_or_dash(df, bid_col, pnl_col):
    """Return Pearson r or '!' if non-significant."""
    df2 = df.dropna(subset=[bid_col, pnl_col])
    if len(df2) < 3:
        return "!"
    
    r, p = scipy.stats.pearsonr(df2[bid_col], df2[pnl_col])
    return f"{r:.3f}" if p < 0.05 else "!"

# horizons you already use
horizons = [0, 5, 10, 300, 600, 1800]

# Create final table structure
corr_table = {
    "Wintermute": {"paid": {}, "top": {}},
    "Selini":     {"paid": {}, "top": {}}
}

for h in horizons:
    pnl_col = f"tb_pnl_t{h}"
    
    # Wintermute
    corr_table["Wintermute"]["paid"][h] = corr_or_dash(wm_round_pnl, "paid_bid_usd", pnl_col)
    corr_table["Wintermute"]["top"][h]  = corr_or_dash(wm_round_pnl, "top_bid_usd",  pnl_col)
    
    # Selini
    corr_table["Selini"]["paid"][h] = corr_or_dash(sel_round_pnl, "paid_bid_usd", pnl_col)
    corr_table["Selini"]["top"][h]  = corr_or_dash(sel_round_pnl, "top_bid_usd",  pnl_col)

# Build LaTeX table string
latex = r"""
\begin{table}[h!]
\centering
\small
\renewcommand{\arraystretch}{1.25}
\begin{tabular}{l|cccccc|cccccc}
\toprule
& \multicolumn{6}{c|}{\textbf{Paid Bid vs PnL (Pearson r)}} 
& \multicolumn{6}{c}{\textbf{Top Bid vs PnL (Pearson r)}} \\
\cmidrule(lr){2-7} \cmidrule(lr){8-13}
\textbf{Entity} 
& 0s & 5s & 10s & 300s & 600s & 1800s
& 0s & 5s & 10s & 300s & 600s & 1800s \\
\midrule
"""

for entity in ["Wintermute", "Selini"]:
    row_paid = [corr_table[entity]["paid"][h] for h in horizons]
    row_top  = [corr_table[entity]["top"][h]  for h in horizons]

    latex += (
        entity + " & " +
        " & ".join(row_paid) + " & " +
        " & ".join(row_top) +
        r" \\" + "\n"
    )

latex += r"""\bottomrule
\end{tabular}
\caption{Correlation between bids (paid and top) and realized PnL across horizons. 
Values show Pearson $r$. ``!'' denotes non-significant results ($p > 0.05$).}
\end{table}
"""

# Save table
with open("bid_pnl_correlation_all_horizons.tex", "w") as f:
    f.write(latex)

print("Saved LaTeX table to bid_pnl_correlation_all_horizons.tex")


In [ ]:
# Ensure consistent columns before concatenation
common_cols = [
    "auction_round",
    "paid_bid_usd",
    "top_bid_usd",
] + [f"tb_pnl_t{h}" for h in horizons]

wm_df = wm_round_pnl[common_cols].copy()
sel_df = sel_round_pnl[common_cols].copy()

combined_round_pnl = pd.concat([wm_df, sel_df], ignore_index=True)

def corr_or_dash(df, bid_col, pnl_col):
    df2 = df.dropna(subset=[bid_col, pnl_col])
    if len(df2) < 3:
        return "–"
    r, p = scipy.stats.pearsonr(df2[bid_col], df2[pnl_col])
    return f"{r:.3f}" if p < 0.05 else "–"

rows = []

for h in horizons:
    pnl_col = f"tb_pnl_t{h}"

    paid_corr = corr_or_dash(combined_round_pnl, "paid_bid_usd", pnl_col)
    top_corr  = corr_or_dash(combined_round_pnl, "top_bid_usd",  pnl_col)

    rows.append({
        "horizon": h,
        "corr_paid": paid_corr,
        "corr_top":  top_corr
    })

combined_corr_df = pd.DataFrame(rows)
print(combined_corr_df)

## Time

In [ ]:
pnl_col = "pnl_t5"   # adjust if needed

def group_by_time_and_tb(df, prefix, pnl_col="pnl_t5"):
    # TODO: Consider only postive PnL trades?
    g = (
        df.groupby(["time_since_round_start", "timeboosted"])
          .agg(
              count=("tx_hash", "size"),
              pnl  =(pnl_col, "sum"),
              vol = ("amount_usd", "sum")
          )
          .reset_index()
          .pivot(
              index="time_since_round_start",
              columns="timeboosted",
              values=["count", "pnl", "vol"]
          )
    )

    # Flatten columns: ("count", True) → "count_True"
    g.columns = [f"{a}_{b}" for a, b in g.columns]

    # Rename logically
    rename_map = {
        "count_True":  f"{prefix}_tb_count",
        "count_False": f"{prefix}_ntb_count",
        "pnl_True":    f"{prefix}_tb_pnl",
        "pnl_False":   f"{prefix}_ntb_pnl",
        "vol_True":    f"{prefix}_tb_vol",
        "vol_False":   f"{prefix}_ntb_vol",
    }
    g = g.rename(columns=rename_map)

    return g.reset_index()

wm_time_tb = group_by_time_and_tb(wm_txs, "wm", pnl_col=pnl_col)
sel_time_tb = group_by_time_and_tb(sel_txs, "sel", pnl_col=pnl_col)

In [ ]:
# --- CONFIG ---
ENTITIES = {
    "WM": {
        "df": wm_time_tb,
        "color": "blue",
        "tb_count": "wm_tb_count",
        "ntb_count": "wm_ntb_count",
        "tb_pnl": "wm_tb_pnl",
        "ntb_pnl": "wm_ntb_pnl",
        "tb_vol": "wm_tb_vol",
        "ntb_vol": "wm_ntb_vol"
    },
    "Sel": {
        "df": sel_time_tb,
        "color": "orange",
        "tb_count": "sel_tb_count",
        "ntb_count": "sel_ntb_count",
        "tb_pnl": "sel_tb_pnl",
        "ntb_pnl": "sel_ntb_pnl",
        "tb_vol": "sel_tb_vol",
        "ntb_vol": "sel_ntb_vol"
    }
}


In [ ]:
def plot_combined(metric, ylabel, title):
    plt.figure(dpi=300)

    for name, cfg in ENTITIES.items():
        df = cfg["df"]
        color = cfg["color"]

        tb_col  = cfg[f"tb_{metric}"]
        ntb_col = cfg[f"ntb_{metric}"]

        # Solid = TB
        plt.plot(df["time_since_round_start"], df[tb_col],
                 label=f"{name} TB", color=color, linestyle="-")
        # Dashed = NTB
        plt.plot(df["time_since_round_start"], df[ntb_col],
                 label=f"{name} non-TB", color=color, linestyle="--")

    plt.xlabel("Time Since Round Start (seconds)")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    # plt.show()
    plt.savefig(f"{metric}_over_time.pdf")


In [ ]:
plot_combined(
    metric="count",
    ylabel="Transaction Count",
    title="Transaction Counts Over Time Since Auction Round Start"
)
plot_combined(
    metric="pnl",
    ylabel="PNL (USD)",
    title="PNL Over Time Since Auction Round Start"
)
plot_combined(
    metric="vol",
    ylabel="Trading Volume (USD)",
    title="USD Volume Over Time Since Auction Round Start"
)


## Reverts

In [ ]:
csv_files = glob.glob("data/reverts/august/*_parsed.csv")
reverts = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

# Prepare times
reverts["block_time"] = pd.to_datetime(reverts["block_time"], utc=True)
auctions["round_start_time"] = pd.to_datetime(auctions["round_start_time"], utc=True)
auctions["round_end_time"]   = pd.to_datetime(auctions["round_end_time"], utc=True)

# Remove timezone for merge_asof
reverts["block_time"]       = reverts["block_time"].dt.tz_convert(None)
auctions["round_start_time"] = auctions["round_start_time"].dt.tz_convert(None)
auctions["round_end_time"]   = auctions["round_end_time"].dt.tz_convert(None)

# Sort
reverts = reverts.sort_values("block_time")
auctions = auctions.sort_values("round_start_time")

# Merge using round start time
reverts = pd.merge_asof(
    reverts,
    auctions[["auction_round", "round_start_time", "round_end_time"]],
    left_on="block_time",
    right_on="round_start_time",
    direction="backward"
)

# Drop rows outside round interval
reverts = reverts[reverts["block_time"] < reverts["round_end_time"]]

reverts['time_since_round_start'] = (reverts['block_time'] - reverts['round_start_time']).dt.total_seconds()


In [ ]:
all_wm_txs = reverts[reverts["tx_to_address"].isin(SEARCHERS["Wintermute"])]
all_sel_txs = reverts[reverts["tx_to_address"].isin(SEARCHERS["Selini"])]

# Find wm txs that are successful and not cex-dex (ie in all_wm_txs but not in wm_txs with success==1)
wm_success_not_cex_dex = all_wm_txs[(all_wm_txs["success"] == 1) & (~all_wm_txs["tx_hash"].isin(wm_txs["tx_hash"]))]
sel_success_not_cex_dex = all_sel_txs[(all_sel_txs["success"] == 1) & (~all_sel_txs["tx_hash"].isin(sel_txs["tx_hash"]))]

wm_tb = all_wm_txs[all_wm_txs["timeboosted"] == True]
wm_non_tb = all_wm_txs[all_wm_txs["timeboosted"] == False]
sel_tb = all_sel_txs[all_sel_txs["timeboosted"] == True]
sel_non_tb = all_sel_txs[all_sel_txs["timeboosted"] == False]

# --- Wintermute ---
wm_cexdex = len(wm_txs)
wm_non_cexdex = len(wm_success_not_cex_dex)

wm_tb_total = len(wm_tb)
wm_tb_success = wm_tb["success"].sum()
wm_tb_rate = wm_tb["success"].mean()

wm_ntb_total = len(wm_non_tb)
wm_ntb_success = wm_non_tb["success"].sum()
wm_ntb_rate = wm_non_tb["success"].mean()

# --- Selini ---
sel_cexdex = len(sel_txs)
sel_non_cexdex = len(sel_success_not_cex_dex)

sel_tb_total = len(sel_tb)
sel_tb_success = sel_tb["success"].sum()
sel_tb_rate = sel_tb["success"].mean()

sel_ntb_total = len(sel_non_tb)
sel_ntb_success = sel_non_tb["success"].sum()
sel_ntb_rate = sel_non_tb["success"].mean()

# Package everything for table
summary = {
    "Wintermute": {
        "cexdex": wm_cexdex,
        "non_cexdex": wm_non_cexdex,
        "tb_total": wm_tb_total,
        "tb_rate": wm_tb_rate,
        "ntb_total": wm_ntb_total,
        "ntb_rate": wm_ntb_rate,
    },
    "Selini": {
        "cexdex": sel_cexdex,
        "non_cexdex": sel_non_cexdex,
        "tb_total": sel_tb_total,
        "tb_rate": sel_tb_rate,
        "ntb_total": sel_ntb_total,
        "ntb_rate": sel_ntb_rate,
    }
}

In [ ]:
import numpy as np

def pct(x, total):
    """Return percentage string like '23.4\\%'."""
    if total == 0:
        return "0.0\\%"
    return f"{(x/total)*100:.1f}\\%"

def fmt(n):
    """Format with thousands separator."""
    return f"{n:,.0f}"

def save_tb_success_table_pretty(all_wm_txs, wm_txs,
                                 all_sel_txs, sel_txs,
                                 outpath="tb_success_table.tex"):

    def compute(all_tx, cexdex):
        total = len(all_tx)
        success = int(all_tx["success"].sum())
        fail = total - success

        tb = all_tx[all_tx["timeboosted"] == True]
        ntb = all_tx[all_tx["timeboosted"] == False]

        tb_total = len(tb)
        tb_success = int(tb["success"].sum())
        tb_fail = tb_total - tb_success

        ntb_total = len(ntb)
        ntb_success = int(ntb["success"].sum())
        ntb_fail = ntb_total - ntb_success

        cexdex_count = len(cexdex)
        cexdex_rate = pct(cexdex_count, success)  # out of successful trades

        return {
            "total": fmt(total),
            "success": pct(success, total),
            "fail": pct(fail, total),

            "cexdex": fmt(cexdex_count),
            "cexdex_rate": cexdex_rate,

            "tb_total": fmt(tb_total),
            "tb_success": pct(tb_success, tb_total),
            "tb_fail": pct(tb_fail, tb_total),

            "ntb_total": fmt(ntb_total),
            "ntb_success": pct(ntb_success, ntb_total),
            "ntb_fail": pct(ntb_fail, ntb_total),
        }

    wm = compute(all_wm_txs, wm_txs)
    sel = compute(all_sel_txs, sel_txs)

    latex = r"""
\begin{table}[ht]
\centering
\begin{tabular}{l rr}
\toprule
\textbf{Metric} & \textbf{Wintermute} & \textbf{Selini} \\
\midrule
\textbf{Total Transactions} &
%(wm_total)s & %(sel_total)s \\
\quad Success Rate &
%(wm_success)s & %(sel_success)s \\
\quad Fail Rate &
%(wm_fail)s & %(sel_fail)s \\
\textbf{CEX--DEX Trades} &
%(wm_cexdex)s & %(sel_cexdex)s \\
\quad Share of Successful &
%(wm_cexdex_rate)s & %(sel_cexdex_rate)s \\
\midrule
\textbf{Timeboosted Transactions} &
%(wm_tb_total)s & %(sel_tb_total)s \\
\quad Success Rate &
%(wm_tb_success)s & %(sel_tb_success)s \\
\quad Fail Rate &
%(wm_tb_fail)s & %(sel_tb_fail)s \\
\addlinespace
\textbf{Non-Timeboosted Transactions} &
%(wm_ntb_total)s & %(sel_ntb_total)s \\
\quad Success Rate &
%(wm_ntb_success)s & %(sel_ntb_success)s \\
\quad Fail Rate &
%(wm_ntb_fail)s & %(sel_ntb_fail)s \\
\bottomrule
\end{tabular}
\caption{Success rates and CEX--DEX shares for Timeboosted and non-Timeboosted transactions of Wintermute and Selini.}
\label{tab:tb-success}
\end{table}
""" % {
        "wm_total": wm["total"], "sel_total": sel["total"],
        "wm_success": wm["success"], "sel_success": sel["success"],
        "wm_fail": wm["fail"], "sel_fail": sel["fail"],

        "wm_cexdex": wm["cexdex"], "sel_cexdex": sel["cexdex"],
        "wm_cexdex_rate": wm["cexdex_rate"], "sel_cexdex_rate": sel["cexdex_rate"],

        "wm_tb_total": wm["tb_total"], "sel_tb_total": sel["tb_total"],
        "wm_tb_success": wm["tb_success"], "sel_tb_success": sel["tb_success"],
        "wm_tb_fail": wm["tb_fail"], "sel_tb_fail": sel["tb_fail"],

        "wm_ntb_total": wm["ntb_total"], "sel_ntb_total": sel["ntb_total"],
        "wm_ntb_success": wm["ntb_success"], "sel_ntb_success": sel["ntb_success"],
        "wm_ntb_fail": wm["ntb_fail"], "sel_ntb_fail": sel["ntb_fail"],
    }

    with open(outpath, "w") as f:
        f.write(latex)

    print(f"✔ LaTeX table saved to: {outpath}")

save_tb_success_table_pretty(
    all_wm_txs, wm_txs,
    all_sel_txs, sel_txs,
    outpath="tb_success_table.tex"
)

In [ ]:
wm_addrs  = SEARCHERS["Wintermute"]
sel_addrs = SEARCHERS["Selini"]

valid_rounds = reverts[
    reverts["tx_to_address"].isin(wm_addrs + sel_addrs)
]["auction_round"].unique()

round_id = 239605 #np.random.choice(valid_rounds)

In [ ]:
def plot_round(round_id):
    # Select all txs for this round
    round_txs = reverts[reverts["auction_round"] == round_id].copy()

    if len(round_txs) == 0:
        print(f"No txs found for round {round_id}")
        return

    # Round metadata
    winner = auctions.loc[auctions["auction_round"] == round_id, "winner_name"].iloc[0]

    # Use precomputed time_since_round_start
    round_txs["t"] = round_txs["time_since_round_start"]

    # Mark whether a tx is a CEX-DEX trade
    round_txs["is_cexdex"] = (
        round_txs["tx_hash"].isin(wm_txs["tx_hash"])
        | round_txs["tx_hash"].isin(sel_txs["tx_hash"])
    )

    # Assign lanes
    def lane(row):
        if row.tx_to_address in SEARCHERS["Wintermute"]:
            return 3 if row.success else 2
        else:
            return 1 if row.success else 0

    round_txs["lane"] = round_txs.apply(lane, axis=1)

    # Entity color
    def entity(addr):
        return "Wintermute" if addr in SEARCHERS["Wintermute"] else "Selini"

    round_txs["entity"] = round_txs["tx_to_address"].apply(entity)
    colors = {"Wintermute": "blue", "Selini": "orange"}

    # Marker mapping:
    #   Revert → "x"
    #   Success CEX-DEX → "*"
    #   Success non-CE-DEX → "o"
    def marker(row):
        if not row.success:
            return "x"
        if row.is_cexdex:
            return "*"        # special marker
        return "o"

    round_txs["marker"] = round_txs.apply(marker, axis=1)

    # Slight jitter for readability
    jitter = np.random.uniform(-0.10, 0.10, size=len(round_txs))

    # --- Plot ---
    plt.figure(dpi=300, figsize=(10, 5))

    for idx, row in enumerate(round_txs.itertuples()):
        plt.scatter(
            row.t,
            row.lane + jitter[idx],
            s=70 if row.is_cexdex else 50,                  # CE-DEX slightly bigger
            color=colors[row.entity],
            marker=row.marker,
            edgecolor="black" if row.is_cexdex else "gray", # strong outline for CE-DEX
            linewidth=1.1,
            alpha=0.85
        )

    # Y axis
    plt.yticks(
        [3, 2, 1, 0],
        [
            "WM — Success",
            "WM — Revert",
            "Selini — Success",
            "Selini — Revert",
        ]
    )

    # Vertical lines each second
    max_t = round_txs["t"].max()
    for sec in range(int(max_t) + 1):
        plt.axvline(sec, color="gray", alpha=0.12, linewidth=0.5)

    plt.xlabel("Seconds Since Round Start")
    plt.title(
        f"Auction Round {round_id} — Winner: {winner}\n"
        f"Blue = WM | Orange = Sel | ○ Success | × Revert | ★ CEX-DEX Success"
    )

    plt.grid(axis="y", alpha=0.35)
    plt.tight_layout()
    plt.savefig(f"auction_round_{round_id}_tx_timeline.pdf")
    plt.show()

plot_round(round_id)